# BiteBuddy Advanced Recommendation Benchmark

This notebook summarizes the advanced offline benchmark run for:

- `bm25`
- `dense_vector`
- `graph_cf`
- `ingredient_hypergraph`
- `advanced_hybrid`
- `neural_reranker`

Notes:

- OpenAI reranking was skipped because `OPENAI_API_KEY` was not set.
- A local neural reranker (`cross-encoder/ms-marco-MiniLM-L-6-v2`) was used instead.
- The split is chronological with the last positive interaction held out per evaluated user.


In [1]:
from pathlib import Path
import json
import pandas as pd

repo_root = Path.cwd()
report_path = repo_root / 'backend' / 'data' / 'processed' / 'recommender_eval_advanced.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
report['config'], report['dataset']

({'sample_users': 100,
  'negatives_per_user': 199,
  'min_positive_interactions': 5,
  'positive_rating_threshold': 4,
  'seed': 42,
  'dense_model': 'sentence-transformers/all-MiniLM-L6-v2',
  'reranker_model': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
  'rerank_top_k': 20,
  'openai_reranker': 'skipped_no_api_key'},
 {'recipes': 231637,
  'interactions': 1132367,
  'positive_interactions': 1003724})

In [2]:
metrics = pd.DataFrame(report['strategies']).T
metrics = metrics[['users_evaluated', 'HitRate@10', 'Recall@10', 'NDCG@10', 'MRR@10', 'CatalogCoverage@10']]
metrics = metrics.sort_values(['NDCG@10', 'MRR@10', 'HitRate@10'], ascending=False)
metrics

,users_evaluated,HitRate@10,Recall@10,NDCG@10,MRR@10,CatalogCoverage@10
popularity,100.0,0.43,0.43,0.2838,0.2375,0.0042
advanced_hybrid,100.0,0.39,0.39,0.2317,0.1826,0.0043
neural_reranker,100.0,0.40,0.40,0.2269,0.1745,0.0043
rating,100.0,0.20,0.20,0.1041,0.0735,0.0042
graph_cf,100.0,0.11,0.11,0.0902,0.0846,0.0043
dense_vector,100.0,0.16,0.16,0.0847,0.0621,0.0043
bm25,100.0,0.15,0.15,0.0758,0.0532,0.0043
ingredient_hypergraph,100.0,0.09,0.09,0.0515,0.0394,0.0043
random,100.0,0.06,0.06,0.0261,0.0158,0.0043


In [3]:
best_ndcg = metrics['NDCG@10'].idxmax()
best_hr = metrics['HitRate@10'].idxmax()
print('Best by NDCG@10:', best_ndcg)
print('Best by HitRate@10:', best_hr)
print('\nDelta vs random baseline:')
random_row = metrics.loc['random']
for name, row in metrics.iterrows():
    if name == 'random':
        continue
    print(f"{name:22s} HR delta={row['HitRate@10'] - random_row['HitRate@10']:+.4f}  NDCG delta={row['NDCG@10'] - random_row['NDCG@10']:+.4f}  MRR delta={row['MRR@10'] - random_row['MRR@10']:+.4f}")

Best by NDCG@10: popularity
Best by HitRate@10: popularity

Delta vs random baseline:
popularity             HR delta=+0.3700  NDCG delta=+0.2577  MRR delta=+0.2217
advanced_hybrid        HR delta=+0.3300  NDCG delta=+0.2056  MRR delta=+0.1668
neural_reranker        HR delta=+0.3400  NDCG delta=+0.2008  MRR delta=+0.1587
rating                 HR delta=+0.1400  NDCG delta=+0.0780  MRR delta=+0.0577
graph_cf               HR delta=+0.0500  NDCG delta=+0.0641  MRR delta=+0.0688
dense_vector           HR delta=+0.1000  NDCG delta=+0.0586  MRR delta=+0.0463
bm25                   HR delta=+0.0900  NDCG delta=+0.0497  MRR delta=+0.0374
ingredient_hypergraph  HR delta=+0.0300  NDCG delta=+0.0254  MRR delta=+0.0236


## Interpretation

What this run says:

- `popularity` is still the strongest single offline baseline on this Food.com holdout split.
- `advanced_hybrid` is the strongest multi-signal strategy we implemented.
- `neural_reranker` slightly improves `HitRate@10` over `advanced_hybrid`, but not `NDCG@10` or `MRR@10`.
- `bm25`, `dense_vector`, `graph_cf`, and `ingredient_hypergraph` are useful components, but not the final winner on their own.

For the product, the recommended deployment logic is still:

1. hard constraint filtering
2. popularity prior
3. hybrid semantic + collaborative reranking
4. optional neural reranking on top candidates


In [4]:
# Optional rerun command (not executed automatically here)
# import subprocess, sys
# subprocess.run([
#     sys.executable,
#     str(repo_root / 'backend' / 'scripts' / 'evaluate_advanced_recommenders.py'),
#     '--sample-users', '100',
#     '--negatives-per-user', '199',
#     '--rerank-top-k', '20',
# ], check=True)